# 🎓 Smart Agri: ฝึกสอนโมเดลปัญญาประดิษฐ์เพื่อการเกษตรและสิ่งแวดล้อม
### 🌿 หลักสูตรปฏิบัติการ Low-Code / No-Code — ฉบับที่ 2 (เพิ่มภาพด้วยเทคนิค Augmentation + การประเมินผลที่น่าเชื่อถือ)

**ผู้จัดทำ:** ชีวะ ทัศนา สาขาวิชาดิจิทัลการเกษตรและสิ่งแวดล้อม คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยราชภัฏรำไพพรรณี

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsanaphy2023/smart-agri-xai-workshop/blob/main/Smart_Agri_AI_LowCode_Workshop.ipynb)

---
**วิธีใช้ (ไม่ต้องเขียนโปรแกรม):** กดปุ่ม Play **[ ▶ ]** ของแต่ละขั้นตอน *ตามลำดับ* และรอให้เสร็จก่อนไปขั้นถัดไป

| ขั้นตอน | งาน | เวลาโดยประมาณ (GPU T4) |
|:--:|:--|:--:|
| 1 | ตรวจเครื่อง + ติดตั้งไลบรารี | 1–2 นาที |
| 2 | เตรียมข้อมูล: กำจัดภาพซ้ำ + แบ่ง train / val / test | 1–3 นาที |
| 3 | **เพิ่มจำนวนภาพ** (หมุน พลิก ซูม ปรับแสง เบลอ สัญญาณรบกวน ฯลฯ) | 1–3 นาที |
| 4 | ฝึกสอนโมเดล + ประเมินผลบนชุดทดสอบ (Precision/Recall/F1, ช่วงเชื่อมั่น 95%) | 3–10 นาที |
| 5 | *(ไม่บังคับ)* Cross-validation เพื่อยืนยันความเสถียรของผล | ≈ 5 × เวลาฝึก |
| 6 | ทดสอบความทนทาน (แสงมืด/สว่าง เบลอ ภาพความละเอียดต่ำ ฯลฯ) | < 1 นาที |
| 7 | ส่งออก ONNX + labels.json + Model Card และดาวน์โหลด | 1 นาที |
| 8 | ทดสอบด้วยภาพของตนเอง (แสดง "ไม่แน่ใจ" เมื่อความมั่นใจต่ำ) | 1 นาที |

> ⚠️ **ก่อนเริ่ม:** *Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save*

> 📌 **หลักการสำคัญ (ทำไมผลจึงน่าเชื่อถือ):**
> 1. **แบ่งข้อมูลก่อนเพิ่มภาพเสมอ** — ภาพที่สร้างเพิ่มใช้กับชุดฝึกเท่านั้น ชุด val/test เป็นภาพจริงล้วน ไม่งั้นตัวเลขจะสูงเกินจริง
> 2. **กำจัดภาพซ้ำ/ใกล้เคียงกันก่อนแบ่ง** เพื่อไม่ให้ภาพเดียวกันอยู่ทั้งชุดฝึกและชุดทดสอบ
> 3. **ชุดทดสอบถูกใช้ครั้งเดียว** ตอนรายงานผลสุดท้าย (ชุด val ใช้เลือกรอบฝึกและปรับค่าความมั่นใจ)
> 4. การเพิ่มภาพ **ไม่ใช่ข้อมูลใหม่** — ช่วยลด overfitting แต่ไม่ทดแทนภาพถ่ายจริงจากพื้นที่

In [ ]:
#@title 🚀 ขั้นตอนที่ 1: ตรวจเครื่อง ติดตั้งไลบรารี และตรวจสอบ GPU { display-mode: "form" }
import importlib, subprocess, sys, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def _ensure(pip_name, module=None):
    try:
        importlib.import_module(module or pip_name)
    except ImportError:
        print(f"📦 กำลังติดตั้ง {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for _pkg in ["kagglehub", "gdown", "onnx", "onnxruntime", "onnxscript"]:
    _ensure(_pkg)

import torch, torchvision

print("=" * 60)
print("🌱 มหาวิทยาลัยราชภัฏรำไพพรรณี | Smart Agri Workshop (v2)")
print("=" * 60)
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
if torch.cuda.is_available():
    print(f"✅ ตรวจพบ GPU: {torch.cuda.get_device_name(0)} — พร้อมฝึกสอนความเร็วสูง")
else:
    print("⚠️ ไม่พบ GPU: โค้ดยังทำงานได้บน CPU แต่ช้ากว่า 10–30 เท่า")
    print("   แก้ไข: Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save แล้วรันขั้นตอนที่ 1 ใหม่")

In [ ]:
#@title 📂 ขั้นตอนที่ 2: เตรียมข้อมูล (ตรวจภาพเสีย + กำจัดภาพซ้ำ + แบ่ง train/val/test) { display-mode: "form" }
import os, csv, shutil, random, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

#@markdown **เลือกแหล่งข้อมูล:**
dataset_choice = "Soil_Types (ดาวน์โหลดอัตโนมัติจาก Kaggle)" #@param ["Soil_Types (ดาวน์โหลดอัตโนมัติจาก Kaggle)", "ZIP จาก Google Drive (ชุดที่วิทยากรแจกในห้องอบรม)", "อัปโหลดไฟล์ ZIP ของตนเอง"]
#@markdown กรณีเลือก *Google Drive*: วางลิงก์หรือรหัสไฟล์ ZIP (ต้องแชร์เป็น "Anyone with the link"):
drive_link_or_id = "" #@param {type:"string"}
#@markdown **วิธีแบ่งข้อมูล:**
split_mode = "รวมทุกภาพแล้วสุ่มแบ่งใหม่ (แนะนำ)" #@param ["รวมทุกภาพแล้วสุ่มแบ่งใหม่ (แนะนำ)", "ใช้การแบ่ง train/val เดิมของ ZIP (กรณีวิทยากรแยกตามแปลงไว้แล้ว)"]
val_ratio = 0.15 #@param {type:"slider", min:0.1, max:0.3, step:0.05}
test_ratio = 0.15 #@param {type:"slider", min:0.1, max:0.3, step:0.05}
#@markdown กำจัดภาพซ้ำ/ใกล้เคียงกันมาก (ป้องกันภาพเดียวกันอยู่ทั้งชุดฝึกและชุดทดสอบ):
remove_duplicates = True #@param {type:"boolean"}

SEED = 42
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RAW, WORK = Path("raw_data"), Path("dataset")

def _imgs(d):
    return sorted(p for p in Path(d).iterdir()
                  if p.is_file() and p.suffix.lower() in IMG_EXT and not p.name.startswith("."))

def _class_dirs(d):
    return sorted(p for p in Path(d).iterdir()
                  if p.is_dir() and not p.name.startswith((".", "__")) and _imgs(p))

def _walk_dirs(base):
    for dp, dns, _ in os.walk(base):
        dns[:] = sorted(d for d in dns if not d.startswith((".", "__")))
        yield Path(dp), dns

def find_dataset_root(base):
    """คืนค่า ('split', train_dir, val_dir) หรือ ('flat', class_root, None)"""
    for dp, dns in _walk_dirs(base):
        low = {d.lower(): d for d in dns}
        vkey = next((k for k in ("val", "valid", "validation", "test") if k in low), None)
        if "train" in low and vkey:
            tr, va = dp / low["train"], dp / low[vkey]
            if len(_class_dirs(tr)) >= 2:
                return "split", tr, va
    for dp, _ in _walk_dirs(base):
        if len(_class_dirs(dp)) >= 2:
            return "flat", dp, None
    return None, None, None

def dhash(path, size=8):
    """ลายนิ้วมือภาพ 64 บิต (difference hash) ใช้หาภาพซ้ำ/ใกล้เคียง"""
    with Image.open(path) as im:
        g = im.convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def fetch_raw():
    if RAW.exists():
        shutil.rmtree(RAW)
    RAW.mkdir()
    if dataset_choice.startswith("Soil_Types"):
        import kagglehub
        print("📥 กำลังดาวน์โหลดจาก Kaggle: prasanshasatpathy/soil-types ...")
        return Path(kagglehub.dataset_download("prasanshasatpathy/soil-types"))
    if dataset_choice.startswith("ZIP จาก Google Drive"):
        import gdown
        src = drive_link_or_id.strip()
        if not src:
            raise RuntimeError("กรุณาวางลิงก์หรือรหัสไฟล์ Google Drive ในช่อง drive_link_or_id แล้วรันใหม่")
        url = src if src.startswith("http") else f"https://drive.google.com/uc?id={src}"
        print("📥 กำลังดาวน์โหลดจาก Google Drive ...")
        out = gdown.download(url, "dataset.zip", quiet=False, fuzzy=True)
        if not out or not zipfile.is_zipfile(out):
            raise RuntimeError("ดาวน์โหลดไม่สำเร็จหรือไม่ใช่ไฟล์ ZIP — ตรวจสอบว่าแชร์เป็น 'Anyone with the link'")
        zip_path = out
    else:
        from google.colab import files
        print("📤 เลือกไฟล์ ZIP (โครงสร้าง: ชื่อคลาส/ภาพ.jpg หรือ train|val/ชื่อคลาส/ภาพ.jpg)")
        up = files.upload()
        zip_path = next((k for k in up if k.lower().endswith(".zip")), None)
        if not zip_path:
            raise RuntimeError("ไม่พบไฟล์ .zip ที่อัปโหลด")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)
    return RAW

try:
    src_dir = fetch_raw()
except Exception as e:
    print(f"❌ เตรียมข้อมูลไม่สำเร็จ: {e}")
    print("💡 ลองเปลี่ยนแหล่งข้อมูลเป็น 'ZIP จาก Google Drive' หรือ 'อัปโหลดไฟล์ ZIP ของตนเอง' แล้วรันใหม่")
    raise

mode, a, b = find_dataset_root(src_dir)
if mode is None:
    raise RuntimeError("ไม่พบโครงสร้างโฟลเดอร์คลาส (ต้องมีอย่างน้อย 2 โฟลเดอร์คลาสที่มีภาพอยู่ข้างใน)")
keep_split = split_mode.startswith("ใช้การแบ่ง") and mode == "split"
if split_mode.startswith("ใช้การแบ่ง") and mode != "split":
    print("ℹ️ ชุดข้อมูลนี้ไม่มีโฟลเดอร์ train/val เดิม → ใช้การสุ่มแบ่งใหม่แทน")

# 1) รวบรวมภาพต่อคลาส (ภาพที่เปิดไม่ได้จะถูกข้าม)
def _ok(p):
    try:
        with Image.open(p) as im:
            im.verify()
        return True
    except Exception:
        return False

pool = {}           # class -> list[(path, origin)]   origin = train|val|flat
bad = 0
roots = [("train", a), ("val", b)] if mode == "split" else [("flat", a)]
for origin, root in roots:
    for cd in _class_dirs(root):
        for p in _imgs(cd):
            if _ok(p):
                pool.setdefault(cd.name, []).append((p, origin))
            else:
                bad += 1
classes = sorted(pool)

# 2) กำจัดภาพซ้ำ/ใกล้เคียง (Hamming distance ≤ 2 จาก 64 บิต) ทั้งภายในและข้ามคลาส
dups = {c: 0 for c in classes}
if remove_duplicates:
    flat_list = [(c, p, o) for c in classes for (p, o) in pool[c]]
    pool = {c: [] for c in classes}
    H, nk = np.zeros((len(flat_list), 64), dtype=bool), 0
    for c, p, o in flat_list:
        h = dhash(p)
        if nk and (np.count_nonzero(H[:nk] != h, axis=1).min() <= 2):
            dups[c] += 1
            continue
        H[nk] = h; nk += 1; pool[c].append((p, o))

# 3) แบ่ง train / val / test (stratified ตามคลาส, สุ่มแบบกำหนด seed)
if WORK.exists():
    shutil.rmtree(WORK)
rng = random.Random(SEED)
manifest = []
for c in classes:
    items = pool[c][:]; rng.shuffle(items)
    if keep_split:
        tr = [x for x in items if x[1] == "train"]
        vv = [x for x in items if x[1] == "val"]
        k = len(vv) // 2
        va, te = vv[k:], vv[:k]          # แบ่งชุด val เดิมครึ่งหนึ่งเป็นชุดทดสอบ
    else:
        n = len(items)
        n_te = max(1, round(n * test_ratio)) if n >= 6 else 0
        n_va = max(1, round(n * val_ratio)) if n >= 6 else 0
        te, va, tr = items[:n_te], items[n_te:n_te + n_va], items[n_te + n_va:]
    for split, lst in (("train", tr), ("val", va), ("test", te)):
        d = WORK / split / c; d.mkdir(parents=True, exist_ok=True)
        for i, (p, o) in enumerate(lst):
            new = d / f"{c}_{split}_{i:04d}{p.suffix.lower()}"
            shutil.copy2(p, new)
            manifest.append((str(new), c, split, p.name))
with open(WORK / "manifest.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["path", "class", "split", "source_name"]); w.writerows(manifest)

# 4) สรุป + เตือนปัญหาที่อาจทำให้ผลไม่น่าเชื่อถือ
def cnt(split, c):
    d = WORK / split / c
    return len(list(d.glob("*"))) if d.exists() else 0

print("\n" + "=" * 66)
print(f"📦 แหล่งข้อมูล: {dataset_choice}")
print(f"🔀 การแบ่ง: {'ใช้ train/val เดิม (แบ่ง val ครึ่งเป็น test)' if keep_split else 'สุ่มแบ่งใหม่ stratified'}")
print(f"{'คลาส':<26}{'train':>7}{'val':>6}{'test':>6}{'ซ้ำที่ตัด':>10}")
tot = [0, 0, 0]
for c in classes:
    r = (cnt("train", c), cnt("val", c), cnt("test", c))
    tot = [x + y for x, y in zip(tot, r)]
    print(f"{c:<26}{r[0]:>7}{r[1]:>6}{r[2]:>6}{dups[c]:>10}")
print(f"{'รวม':<26}{tot[0]:>7}{tot[1]:>6}{tot[2]:>6}{sum(dups.values()):>10}")
if bad:
    print(f"⚠️ ข้ามไฟล์ภาพที่เสียหาย {bad} ไฟล์")
warn = [c for c in classes if cnt("train", c) < 5 or cnt("val", c) < 1 or cnt("test", c) < 1]
if warn:
    print(f"⚠️ คลาสที่มีภาพน้อยเกินไปสำหรับการแบ่ง: {warn} → ควรเพิ่มภาพจริง (อย่างน้อยคลาสละ 30–50 ภาพ)")
if tot[2] < 30:
    print(f"⚠️ ชุดทดสอบมีเพียง {tot[2]} ภาพ: ตัวเลขความแม่นยำจะมีช่วงความคลาดเคลื่อนกว้าง (ดูช่วงเชื่อมั่นในขั้นตอนที่ 4)")
if not warn:
    print("✅ ชุดข้อมูลพร้อม")

n = len(classes)
fig, axs = plt.subplots(1, n, figsize=(2.6 * n, 2.8)); axs = [axs] if n == 1 else axs
for i, (ax, c) in enumerate(zip(axs, classes)):
    ax.imshow(Image.open(sorted((WORK / "train" / c).glob("*"))[0]).convert("RGB")); ax.set_title(f"class {i}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("ความหมายของเลขคลาส:", {i: c for i, c in enumerate(classes)})

In [ ]:
#@title 🔄 ขั้นตอนที่ 3: เพิ่มจำนวนภาพฝึกสอน (หมุน พลิก ซูม ปรับแสง เบลอ สัญญาณรบกวน) { display-mode: "form" }
import io, random, shutil
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageFilter, ImageDraw
from torchvision import transforms as T
import matplotlib.pyplot as plt

#@markdown เพิ่มภาพ **เฉพาะชุดฝึก (train)** จนแต่ละคลาสมีจำนวนภาพรวมตามเป้าหมาย (ชุด val/test ไม่ถูกแตะ) — ค่า 0 = ไม่เพิ่ม:
target_per_class = 300 #@param [0, 100, 200, 300, 500, 1000]
aug_strength = "ปานกลาง" #@param ["เบา", "ปานกลาง", "แรง"]
#@markdown พลิกแนวตั้ง (เหมาะกับภาพดินหรือใบที่ไม่มีทิศทางตายตัว; ปิดถ้าภาพมีด้านบน-ล่างชัดเจน):
vertical_flip = True #@param {type:"boolean"}

S = {"เบา": 0.5, "ปานกลาง": 1.0, "แรง": 1.5}[aug_strength]
AUG_SIZE = 320
AUG_DIR = Path("dataset/train_aug")
if AUG_DIR.exists():
    shutil.rmtree(AUG_DIR)

def _mean_color(im):
    return tuple(int(v) for v in np.asarray(im.resize((16, 16))).reshape(-1, 3).mean(0))

def augment_once(im, rnd):
    """สุ่มชุดเทคนิค 4–7 อย่างต่อภาพ (สีถูกปรับเบามาก เพราะสีคือคุณลักษณะวินิจฉัยของดิน/ใบ)"""
    fill = _mean_color(im)
    tf = [T.RandomHorizontalFlip(0.5)]
    if vertical_flip:
        tf.append(T.RandomVerticalFlip(0.5))
    tf += [
        T.RandomAffine(degrees=30 * S, translate=(0.1 * S, 0.1 * S),
                       scale=(max(0.6, 1 - 0.25 * S), 1 + 0.25 * S), shear=10 * S, fill=fill),   # หมุน/เลื่อน/ซูม/เฉือน
        T.RandomResizedCrop((AUG_SIZE, AUG_SIZE), scale=(max(0.4, 1 - 0.45 * S), 1.0), ratio=(0.85, 1.18)),  # ครอป
        T.ColorJitter(brightness=0.3 * S, contrast=0.3 * S, saturation=0.2 * S, hue=0.02 * S),     # แสง/คอนทราสต์
    ]
    if rnd.random() < 0.25:
        tf.append(T.RandomPerspective(distortion_scale=0.25 * S, p=1.0, fill=fill))              # เปลี่ยนมุมกล้อง
    im = T.Compose(tf)(im)
    if rnd.random() < 0.3:
        im = im.filter(ImageFilter.GaussianBlur(rnd.uniform(0.5, 1.5)))                          # เบลอ (โฟกัสไม่ชัด)
    if rnd.random() < 0.3:
        arr = np.asarray(im, dtype=np.float32)
        arr += np.random.normal(0, rnd.uniform(4, 12) * S, arr.shape)                            # สัญญาณรบกวนเซนเซอร์
        im = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if rnd.random() < 0.3:
        buf = io.BytesIO(); im.save(buf, "JPEG", quality=rnd.randint(35, 75)); buf.seek(0)       # บีบอัด JPEG
        im = Image.open(buf).convert("RGB")
    if rnd.random() < 0.25:
        w, h = im.size; cw, ch = int(w * rnd.uniform(0.1, 0.25)), int(h * rnd.uniform(0.1, 0.25))
        x0, y0 = rnd.randint(0, w - cw), rnd.randint(0, h - ch)
        ImageDraw.Draw(im).rectangle([x0, y0, x0 + cw, y0 + ch], fill=fill)                      # บดบังบางส่วน (Cutout)
    return im

rnd = random.Random(SEED)
torch.manual_seed(SEED); np.random.seed(SEED)
classes_aug = sorted(p.name for p in Path("dataset/train").iterdir() if p.is_dir())
summary = []
if target_per_class > 0:
    for c in classes_aug:
        src = sorted(p for p in (Path("dataset/train") / c).iterdir() if p.suffix.lower() in IMG_EXT)
        need = max(0, target_per_class - len(src))
        out = AUG_DIR / c; out.mkdir(parents=True, exist_ok=True)
        for k in range(need):
            sp = src[k % len(src)]                       # วนใช้ภาพต้นฉบับทีละใบ ให้ครอบคลุมเท่า ๆ กัน
            im = Image.open(sp).convert("RGB")
            augment_once(im, rnd).save(out / f"{sp.stem}_aug{k:04d}.jpg", quality=92)
        summary.append((c, len(src), need))
    print("=" * 56)
    print(f"{'คลาส':<26}{'ภาพจริง':>8}{'สร้างเพิ่ม':>10}{'รวม':>8}")
    for c, n0, n1 in summary:
        print(f"{c:<26}{n0:>8}{n1:>10}{n0+n1:>8}")
    print(f"✅ ชุดฝึกรวม {sum(s[1]+s[2] for s in summary)} ภาพ (จริง {sum(s[1] for s in summary)} + สร้างเพิ่ม {sum(s[2] for s in summary)})")
    if any(s[1] >= target_per_class for s in summary):
        print("ℹ️ บางคลาสมีภาพจริงถึงเป้าหมายแล้ว จึงไม่สร้างเพิ่ม")
    ratio = max(s[2] for s in summary) / max(1, min(s[1] for s in summary))
    if ratio > 10:
        print("⚠️ สร้างเพิ่มมากกว่า 10 เท่าของภาพจริงในบางคลาส: ภาพจะคล้ายกันมากและโมเดลอาจ overfit — ควรเพิ่มภาพจริงแทนการเพิ่มเป้าหมาย")
else:
    print("ℹ️ ไม่ได้เพิ่มภาพ (target_per_class = 0)")

# ตัวอย่าง: ภาพจริง 1 ใบ + ภาพที่สร้างเพิ่ม 7 ใบ
c0 = classes_aug[0]
sp = sorted((Path("dataset/train") / c0).iterdir())[0]
base = Image.open(sp).convert("RGB")
fig, axs = plt.subplots(1, 8, figsize=(18, 2.6))
axs[0].imshow(base); axs[0].set_title("original"); axs[0].axis("off")
for i in range(1, 8):
    axs[i].imshow(augment_once(base, random.Random(100 + i))); axs[i].set_title(f"aug {i}"); axs[i].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
#@title ⚙️ ขั้นตอนที่ 4: ฝึกสอนโมเดล + ประเมินผลบนชุดทดสอบ { display-mode: "form" }
import copy, json, time, random, math
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

#@markdown **การตั้งค่าพารามิเตอร์แบบ Low-Code:**
model_type = "MobileNetV3 Small (ขนาดเบา เหมาะกับสมาร์ทโฟนและเว็บ)" #@param ["MobileNetV3 Small (ขนาดเบา เหมาะกับสมาร์ทโฟนและเว็บ)", "ResNet18 (โมเดลมาตรฐาน)"]
epochs = 15 #@param {type:"slider", min:5, max:40, step:5}
batch_size = 16 #@param [8, 16, 32]
learning_rate = 0.001 #@param [0.0001, 0.0005, 0.001, 0.005]
online_augment = True #@param {type:"boolean"}

SEED = 42
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
arch = "mobilenet_v3_small" if model_type.startswith("MobileNet") else "resnet18"
IMG_SIZE, MEAN, STD = 224, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
workers = 2 if device.type == "cuda" else 0

def build_model(arch, num_classes, pretrained=True):
    if arch == "mobilenet_v3_small":
        W, ctor = models.MobileNet_V3_Small_Weights.DEFAULT, models.mobilenet_v3_small
    else:
        W, ctor = models.ResNet18_Weights.DEFAULT, models.resnet18
    try:
        m = ctor(weights=W if pretrained else None)
    except Exception as e:
        print(f"⚠️ โหลดน้ำหนัก ImageNet ไม่ได้ ({type(e).__name__}) → ใช้น้ำหนักสุ่ม ความแม่นยำจะต่ำลงมาก (ตรวจสอบอินเทอร์เน็ต)")
        m = ctor(weights=None)
    if arch == "mobilenet_v3_small":
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

tf_val = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
                             transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_online = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_train = tf_online if online_augment else tf_val

class Samples(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def list_items(split, classes):
    out = []
    for ci, c in enumerate(classes):
        d = Path("dataset") / split / c
        if d.exists():
            out += [(str(p), ci) for p in sorted(d.iterdir()) if p.suffix.lower() in IMG_EXT]
    return out

def make_loader(items, tf, shuffle):
    ds = Samples(items, tf)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=workers,
                      pin_memory=device.type == "cuda", drop_last=shuffle and len(ds) % batch_size == 1)

@torch.no_grad()
def predict(net, loader):
    net.eval(); L, Y = [], []
    for x, y in loader:
        L.append(net(x.to(device)).cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def run_training(train_items, val_items, select_best=True, verbose=True, tf=None, n_epochs=None):
    n_epochs = n_epochs or epochs
    net = build_model(arch, n_cls).to(device)
    cnt = np.bincount([y for _, y in train_items], minlength=n_cls).astype(float)
    w = None
    if cnt.min() > 0 and cnt.max() / cnt.min() > 1.5:       # ชุดฝึกไม่สมดุล → ถ่วงน้ำหนักคลาส
        w = torch.tensor(cnt.sum() / (n_cls * cnt), dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    tl = make_loader(train_items, tf or tf_train, True)
    vl = make_loader(val_items, tf_val, False) if val_items else None
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
    best_acc, best_loss, best_state, best_ep = -1.0, 1e9, None, n_epochs
    for ep in range(1, n_epochs + 1):
        net.train(); ls = co = sn = 0
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); out = net(x); loss = crit(out, y); loss.backward(); opt.step()
            ls += loss.item() * x.size(0); co += (out.argmax(1) == y).sum().item(); sn += x.size(0)
        sch.step()
        hist["tr_loss"].append(ls / sn); hist["tr_acc"].append(100.0 * co / sn)
        msg = f"Epoch {ep:02d}/{n_epochs:02d} | train loss {ls/sn:.4f} acc {100.0*co/sn:6.2f}%"
        if vl is not None:
            lg, yy = predict(net, vl)
            vloss, vacc = F.cross_entropy(lg, yy).item(), 100.0 * (lg.argmax(1) == yy).float().mean().item()
            hist["va_loss"].append(vloss); hist["va_acc"].append(vacc)
            msg += f" | val loss {vloss:.4f} acc {vacc:6.2f}%"
            if select_best and (vacc > best_acc or (vacc == best_acc and vloss < best_loss)):
                best_acc, best_loss, best_ep, best_state = vacc, vloss, ep, copy.deepcopy(net.state_dict()); msg += " ⭐"
        if verbose:
            print(msg)
    if best_state is not None:
        net.load_state_dict(best_state)
    return net.eval(), hist, best_ep

def metrics(y_true, y_pred, n):
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros(n), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(n), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0)
    present = cm.sum(1) > 0
    return {"cm": cm, "precision": prec, "recall": rec, "f1": f1, "support": cm.sum(1),
            "acc": 100.0 * tp.sum() / max(1, cm.sum()), "macro_f1": 100.0 * f1[present].mean()}

def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * max(0, c - h), 100 * min(1, c + h)

def fit_temperature(logits, y):
    if len(y) < 10: return 1.0
    grid = np.linspace(0.25, 5.0, 96)
    nll = [F.cross_entropy(logits / t, y).item() for t in grid]
    return float(grid[int(np.argmin(nll))])

# ── เตรียมข้อมูล ──
classes = sorted(p.name for p in Path("dataset/train").iterdir() if p.is_dir())
n_cls = len(classes)
orig_items = list_items("train", classes)
aug_items = list_items("train_aug", classes)
tr_items, va_items, te_items = orig_items + aug_items, list_items("val", classes), list_items("test", classes)
assert len(va_items) > 0 and len(te_items) > 0, "ชุด val/test ว่าง — เพิ่มจำนวนภาพหรือปรับสัดส่วนในขั้นตอนที่ 2"

print(f"🚀 {model_type} | อุปกรณ์: {device} | Epochs={epochs} Batch={batch_size} LR={learning_rate} | augment ระหว่างฝึก={online_augment}")
print(f"📦 {n_cls} คลาส | train {len(tr_items)} (จริง {len(orig_items)} + สร้างเพิ่ม {len(aug_items)}) | val {len(va_items)} | test {len(te_items)}")
print("=" * 70)
t0 = time.time()
net, hist, best_ep = run_training(tr_items, va_items)
print("=" * 70)
print(f"⏱️ ใช้เวลา {time.time()-t0:.0f} วินาที | เลือกโมเดลรอบที่ {best_ep} (จากความแม่นยำ/loss บนชุด val)")

# ── ปรับเทียบความมั่นใจ (Temperature scaling) ด้วยชุด val ──
lg_val, y_val = predict(net, make_loader(va_items, tf_val, False))
TEMP = fit_temperature(lg_val, y_val)

# ── ประเมินผลสุดท้ายบนชุดทดสอบ (ใช้ครั้งเดียว) ──
lg_te, y_te = predict(net, make_loader(te_items, tf_val, False))
pred_te = lg_te.argmax(1).numpy()
M = metrics(y_te.numpy(), pred_te, n_cls)
k_ok = int((pred_te == y_te.numpy()).sum()); n_te = len(y_te)
ci_lo, ci_hi = wilson(k_ok, n_te)
conf = F.softmax(lg_te / TEMP, 1).max(1).values.numpy()

print(f"\n🎯 ผลบนชุดทดสอบ (ภาพจริงที่โมเดลไม่เคยเห็น, n={n_te})")
print(f"   Accuracy  = {M['acc']:.2f}%   (ช่วงเชื่อมั่น 95%: {ci_lo:.1f}–{ci_hi:.1f}%)")
print(f"   Macro-F1  = {M['macro_f1']:.2f}%")
print(f"   ความมั่นใจเฉลี่ยของโมเดล = {100*conf.mean():.1f}% (T={TEMP:.2f}) เทียบกับความแม่นยำจริง {M['acc']:.1f}%")
print(f"\n{'คลาส':<26}{'Precision':>10}{'Recall':>9}{'F1':>8}{'จำนวน':>7}")
for i, c in enumerate(classes):
    print(f"{c:<26}{100*M['precision'][i]:>9.1f}%{100*M['recall'][i]:>8.1f}%{100*M['f1'][i]:>7.1f}%{M['support'][i]:>7}")
if n_te < 100:
    print(f"\nℹ️ ชุดทดสอบมี {n_te} ภาพ ช่วงเชื่อมั่นจึงกว้าง ({ci_hi-ci_lo:.0f} จุดเปอร์เซ็นต์) — อย่าอ้างความแม่นยำเป็นตัวเลขเดียว; ใช้ขั้นตอนที่ 5 (Cross-validation) ยืนยัน")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title("Accuracy (%)"); ax[1].legend()
ax[2].imshow(M["cm"], cmap="Blues"); ax[2].set_title("Confusion matrix (test)")
ax[2].set_xlabel("predicted class"); ax[2].set_ylabel("true class")
ax[2].set_xticks(range(n_cls)); ax[2].set_yticks(range(n_cls))
for i in range(n_cls):
    for j in range(n_cls):
        ax[2].text(j, i, M["cm"][i, j], ha="center", va="center", color="white" if M["cm"][i, j] > M["cm"].max() / 2 else "black")
plt.tight_layout(); plt.show()
print("ความหมายของเลขคลาส:", {i: c for i, c in enumerate(classes)})

REPORT = {"test_acc": round(float(M["acc"]), 2), "test_ci95": [round(float(ci_lo), 1), round(float(ci_hi), 1)], "test_macro_f1": round(float(M["macro_f1"]), 2),
          "n_test": int(n_te), "n_val": len(va_items), "n_train_real": len(orig_items), "n_train_augmented": len(aug_items),
          "per_class": {c: {"precision": round(float(M["precision"][i]), 3), "recall": round(float(M["recall"][i]), 3),
                            "f1": round(float(M["f1"][i]), 3), "support": int(M["support"][i])} for i, c in enumerate(classes)},
          "epochs": int(epochs), "best_epoch": int(best_ep), "learning_rate": float(learning_rate), "temperature": round(float(TEMP), 3)}
torch.save({"arch": arch, "class_names": classes, "state_dict": {k: v.cpu() for k, v in net.state_dict().items()},
            "temperature": TEMP, "report": REPORT, "img_size": IMG_SIZE, "mean": MEAN, "std": STD}, "smart_agri_model.pt")
print("💾 บันทึก smart_agri_model.pt แล้ว")

In [ ]:
#@title 🔁 ขั้นตอนที่ 5 (ไม่บังคับ): Cross-validation เพื่อยืนยันความเสถียรของผล { display-mode: "form" }
import gc
#@markdown จำนวนส่วน (folds) — ฝึกซ้ำ k รอบ โดยแต่ละรอบเก็บข้อมูลจริง 1 ส่วนไว้ทดสอบ; 0 = ข้ามขั้นตอนนี้:
cv_folds = 5 #@param [0, 3, 5]

if cv_folds == 0:
    print("ℹ️ ข้ามขั้นตอน Cross-validation")
else:
    items = list_items("train", classes) + list_items("val", classes)      # ภาพจริงเท่านั้น (ไม่รวมภาพสร้างเพิ่ม และไม่รวมชุด test)
    rngc = random.Random(SEED)
    folds = [[] for _ in range(cv_folds)]
    for ci in range(n_cls):
        ci_items = [it for it in items if it[1] == ci]; rngc.shuffle(ci_items)
        for j, it in enumerate(ci_items):
            folds[j % cv_folds].append(it)
    print(f"🔁 {cv_folds}-fold CV บนภาพจริง {len(items)} ภาพ | ฝึก {epochs} epochs ต่อรอบ (ใช้ augment ระหว่างฝึกเท่านั้น เพื่อไม่ให้ภาพสร้างเพิ่มรั่วเข้าชุดทดสอบของรอบนั้น)")
    accs, f1s, allp, ally = [], [], [], []
    t0 = time.time()
    for k in range(cv_folds):
        te_k = folds[k]; tr_k = [it for j, f in enumerate(folds) if j != k for it in f]
        net_k, _, _ = run_training(tr_k, None, select_best=False, verbose=False, tf=tf_online)
        lg, yy = predict(net_k, make_loader(te_k, tf_val, False))
        mk = metrics(yy.numpy(), lg.argmax(1).numpy(), n_cls)
        accs.append(mk["acc"]); f1s.append(mk["macro_f1"]); allp += lg.argmax(1).tolist(); ally += yy.tolist()
        print(f"  fold {k+1}/{cv_folds}: acc {mk['acc']:6.2f}% | macro-F1 {mk['macro_f1']:6.2f}% | n={len(te_k)}")
        del net_k; gc.collect(); torch.cuda.empty_cache()
    mp = metrics(np.array(ally), np.array(allp), n_cls)
    print("=" * 60)
    print(f"Accuracy  = {np.mean(accs):.2f}% ± {np.std(accs, ddof=1):.2f} (SD)   ช่วง {min(accs):.1f}–{max(accs):.1f}%")
    print(f"Macro-F1  = {np.mean(f1s):.2f}% ± {np.std(f1s, ddof=1):.2f} (SD)")
    print(f"Recall รายคลาส (รวมทุก fold): " + ", ".join(f"{c}={100*r:.0f}%" for c, r in zip(classes, mp["recall"])))
    print(f"⏱️ ใช้เวลา {time.time()-t0:.0f} วินาที")
    if np.std(accs, ddof=1) > 8:
        print("⚠️ ผลแต่ละ fold ต่างกันมาก → ผลไม่เสถียร: ข้อมูลน้อยหรือไม่สมดุล ควรเพิ่มภาพจริงก่อนนำไปใช้")
    REPORT["cv"] = {"folds": cv_folds, "acc_mean": round(float(np.mean(accs)), 2), "acc_sd": round(float(np.std(accs, ddof=1)), 2),
                    "macro_f1_mean": round(float(np.mean(f1s)), 2)}

In [ ]:
#@title 🧪 ขั้นตอนที่ 6: ทดสอบความทนทานต่อสภาพภาพถ่ายภาคสนาม { display-mode: "form" }
import io
from PIL import ImageEnhance, ImageFilter

def _noise(im):
    a = np.asarray(im, dtype=np.float32) + np.random.RandomState(0).normal(0, 15, (im.size[1], im.size[0], 3))
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
def _jpeg(im):
    b = io.BytesIO(); im.save(b, "JPEG", quality=15); b.seek(0); return Image.open(b).convert("RGB")
def _lowres(im):
    w, h = im.size; return im.resize((max(1, w // 6), max(1, h // 6))).resize((w, h))

PERT = {
    "ต้นฉบับ": lambda im: im,
    "มืดลง (แสงน้อย)": lambda im: ImageEnhance.Brightness(im).enhance(0.55),
    "สว่างจัด (แดดจ้า)": lambda im: ImageEnhance.Brightness(im).enhance(1.6),
    "คอนทราสต์ต่ำ": lambda im: ImageEnhance.Contrast(im).enhance(0.55),
    "เบลอ (โฟกัสไม่ชัด)": lambda im: im.filter(ImageFilter.GaussianBlur(2.5)),
    "สัญญาณรบกวน": _noise,
    "หมุน 25°": lambda im: im.rotate(25, resample=Image.BILINEAR),
    "บีบอัด JPEG หนัก": _jpeg,
    "ความละเอียดต่ำ (กล้องเก่า)": _lowres,
}
net.eval(); base_acc = None; rows = []
for name, fn in PERT.items():
    xs = torch.stack([tf_val(fn(Image.open(p).convert("RGB"))) for p, _ in te_items])
    with torch.no_grad():
        pr = torch.cat([net(xs[i:i + 32].to(device)).argmax(1).cpu() for i in range(0, len(xs), 32)])
    acc = 100.0 * (pr.numpy() == np.array([y for _, y in te_items])).mean()
    base_acc = acc if base_acc is None else base_acc
    rows.append((name, acc, acc - base_acc))
print(f"{'เงื่อนไข':<30}{'Accuracy':>10}{'เปลี่ยนแปลง':>14}")
for name, acc, d in rows:
    flag = "  ⚠️" if d < -10 else ""
    print(f"{name:<30}{acc:>9.1f}%{d:>+13.1f}{flag}")
worst = min(rows[1:], key=lambda r: r[1])
print(f"\nสภาพที่โมเดลอ่อนที่สุด: {worst[0]} ({worst[1]:.1f}%)")
if worst[2] < -10:
    print("⚠️ ความแม่นยำลดลงมากเมื่อสภาพภาพต่างจากชุดฝึก — ภาคสนามควรถ่ายภาพในสภาพแสงที่ควบคุม (เช่น ใช้ร่มบังแดด/แผ่นอ้างอิงสีเทา) และเพิ่มภาพจริงในสภาพดังกล่าวเข้าชุดฝึก")
REPORT["robustness"] = {n: round(float(a), 1) for n, a, _ in rows}

In [ ]:
#@title ⚡ ขั้นตอนที่ 7: ส่งออก ONNX ตรวจสอบความถูกต้อง สร้าง Model Card และดาวน์โหลด { display-mode: "form" }
import os, json, zipfile
import numpy as np
import torch, onnx, onnxruntime as ort

#@markdown ประเภทงานของโมเดล (ใช้บอกระบบปลายทาง):
task_type = "soil" #@param ["soil", "plant"]
#@markdown ความมั่นใจขั้นต่ำ — ต่ำกว่านี้ให้ระบบแสดง "ไม่แน่ใจ" แทนการฟันธง:
unsure_threshold = 0.60 #@param {type:"slider", min:0.4, max:0.9, step:0.05}

ckpt = torch.load("smart_agri_model.pt", map_location="cpu")
export_net = build_model(ckpt["arch"], len(ckpt["class_names"]), pretrained=False)
export_net.load_state_dict(ckpt["state_dict"])
export_net.eval().cpu()

onnx_path = "smart_agri_model.onnx"
dummy = torch.randn(1, 3, ckpt["img_size"], ckpt["img_size"])
kw = dict(input_names=["input"], output_names=["output"],
          dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}, opset_version=17)
try:
    torch.onnx.export(export_net, dummy, onnx_path, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(export_net, dummy, onnx_path, **kw)
onnx.checker.check_model(onnx.load(onnx_path))

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
with torch.no_grad():
    ref = export_net(dummy).numpy()
diff = float(np.abs(ref - sess.run(None, {"input": dummy.numpy()})[0]).max())
assert diff < 1e-3, f"ผลลัพธ์ ONNX ไม่ตรงกับ PyTorch (ต่างสูงสุด {diff:.5f})"

rep = ckpt["report"]
meta = {"task": task_type, "architecture": ckpt["arch"], "classes": ckpt["class_names"],
        "input": {"name": "input", "shape": [1, 3, ckpt["img_size"], ckpt["img_size"]], "layout": "NCHW", "dtype": "float32"},
        "preprocess": {"resize_shorter_side": 256, "center_crop": ckpt["img_size"], "scale": "pixel/255",
                       "mean": ckpt["mean"], "std": ckpt["std"], "color": "RGB"},
        "output": "logits → softmax(logits / temperature)",
        "temperature": ckpt["temperature"], "unsure_threshold": float(unsure_threshold),
        "evaluation": {k: rep[k] for k in ("test_acc", "test_ci95", "test_macro_f1", "n_test", "per_class")},
        "robustness": REPORT.get("robustness"), "cross_validation": REPORT.get("cv")}
with open("labels.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

cv = REPORT.get("cv")
card = f"""# Model Card — Smart Agri ({task_type})
- สถาปัตยกรรม: {ckpt['arch']} (Transfer Learning จาก ImageNet) | คลาส ({len(meta['classes'])}): {', '.join(meta['classes'])}
- ข้อมูลฝึก: ภาพจริง {rep['n_train_real']} + สร้างเพิ่มด้วย augmentation {rep['n_train_augmented']} | val {rep['n_val']} | test {rep['n_test']} (ภาพจริงล้วน)
- ผลบนชุดทดสอบ: Accuracy {rep['test_acc']}% (ช่วงเชื่อมั่น 95%: {rep['test_ci95'][0]}–{rep['test_ci95'][1]}%), Macro-F1 {rep['test_macro_f1']}%
- Cross-validation: {('%d-fold, Accuracy %.1f%% ± %.1f' % (cv['folds'], cv['acc_mean'], cv['acc_sd'])) if cv else 'ไม่ได้ทำ'}
- ความมั่นใจปรับเทียบด้วย temperature = {ckpt['temperature']:.2f}; ต่ำกว่า {unsure_threshold:.0%} ให้แสดง "ไม่แน่ใจ"

## ข้อจำกัด (ต้องอ่านก่อนใช้งาน)
- เป็นเครื่องมือ **คัดกรองเบื้องต้น** ไม่ใช่ผลวินิจฉัยทางห้องปฏิบัติการ ต้องยืนยันด้วยการวัดจริง (เช่น pH, N-P-K) หรือผู้เชี่ยวชาญ
- เรียนรู้จากภาพชุดนี้เท่านั้น ภาพจากพื้นที่/กล้อง/สภาพแสงอื่นอาจแม่นยำต่ำกว่าตัวเลขข้างต้น
- ภาพที่สร้างเพิ่มไม่ใช่ข้อมูลใหม่ ไม่ทดแทนภาพถ่ายจริง
"""
with open("MODEL_CARD.md", "w", encoding="utf-8") as f:
    f.write(card)

pkg = "smart_agri_model_package.zip"
with zipfile.ZipFile(pkg, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in (onnx_path, "labels.json", "MODEL_CARD.md", "smart_agri_model.pt"):
        z.write(fn)

print(f"✅ ส่งออก ONNX สำเร็จ: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f} MB) | ตรวจเทียบกับ PyTorch ผ่าน (ต่างสูงสุด {diff:.2e})")
print(f"📝 labels.json (คลาส, วิธีเตรียมภาพ, temperature, ค่าประเมิน) + MODEL_CARD.md")
print(f"📦 แพ็กเกจ: {pkg} ({os.path.getsize(pkg)/1e6:.1f} MB)")
try:
    from google.colab import files
    print("📥 กำลังดาวน์โหลดแพ็กเกจลงเครื่อง (หากเบราว์เซอร์ถามสิทธิ์ ให้กด 'อนุญาต') ...")
    files.download(pkg)
except Exception:
    print(f"💾 ไม่ได้รันบน Colab: ไฟล์อยู่ที่ {os.path.abspath(pkg)}")

In [ ]:
#@title 🔬 ขั้นตอนที่ 8: ทดสอบโมเดลด้วยภาพของตนเอง { display-mode: "form" }
import glob, json, os, random
import numpy as np
import onnxruntime as ort
from PIL import Image
import matplotlib.pyplot as plt

#@markdown เลือกภาพที่ใช้ทดสอบ (แนะนำ "อัปโหลดภาพของฉัน" — ถ่ายดินหรือใบพืชจริงจากมือถือ):
test_source = "สุ่มภาพจากชุดทดสอบ" #@param ["อัปโหลดภาพของฉัน", "สุ่มภาพจากชุดทดสอบ"]

meta = json.load(open("labels.json", encoding="utf-8"))
classes, pp = meta["classes"], meta["preprocess"]
T_, thr = meta["temperature"], meta["unsure_threshold"]
sess = ort.InferenceSession("smart_agri_model.onnx", providers=["CPUExecutionProvider"])

def preprocess(path):
    im = Image.open(path).convert("RGB")
    w, h = im.size; s = pp["resize_shorter_side"] / min(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    w, h = im.size; c = pp["center_crop"]; l, t = (w - c) // 2, (h - c) // 2
    im = im.crop((l, t, l + c, t + c))
    x = (np.asarray(im, dtype=np.float32) / 255.0 - np.array(pp["mean"], np.float32)) / np.array(pp["std"], np.float32)
    return im, x.transpose(2, 0, 1)[None].astype(np.float32)

if test_source.startswith("อัปโหลด"):
    from google.colab import files
    paths = list(files.upload().keys())
else:
    pool_ = glob.glob("dataset/test/*/*")
    paths = random.sample(pool_, k=min(3, len(pool_)))

for p in paths:
    try:
        im, x = preprocess(p)
    except Exception as e:
        print(f"⚠️ เปิดไฟล์ {p} ไม่ได้: {e}"); continue
    z = sess.run(None, {"input": x})[0][0] / T_
    prob = np.exp(z - z.max()); prob /= prob.sum()
    top = prob.argsort()[::-1][:3]
    plt.figure(figsize=(3, 3)); plt.imshow(im); plt.axis("off"); plt.show()
    print(f"📷 {os.path.basename(p)}")
    if prob[top[0]] < thr:
        print(f"   ❓ ไม่แน่ใจ (ความมั่นใจสูงสุด {prob[top[0]]*100:.0f}% < เกณฑ์ {thr*100:.0f}%) — ถ่ายใหม่ในแสงสม่ำเสมอ หรือส่งตรวจยืนยัน")
    for r, i in enumerate(top, 1):
        print(f"   {r}. {classes[i]:<28} {prob[i]*100:5.1f}%")
    print("   ℹ️ ผลเป็นการคัดกรองเบื้องต้น ควรยืนยันด้วยการตรวจวัดจริง")

---
### 🎯 สิ่งที่ต้องทำต่อไปใน Workshop
1. แตกไฟล์ `smart_agri_model_package.zip` จะได้ `smart_agri_model.onnx`, `labels.json`, `MODEL_CARD.md`
2. นำ `.onnx` + `labels.json` ไปใช้กับ Streamlit Web Dashboard (ใช้ `temperature` และ `unsure_threshold` ใน `labels.json` เมื่อแปลผล)
3. ถ่ายภาพดินหรือใบพืชจริงเพื่อทดสอบ และเก็บภาพที่โมเดลทำนายผิดไว้เพิ่มเข้าชุดฝึกรอบถัดไป

### 🛠️ แก้ปัญหาเบื้องต้น
| อาการ | สาเหตุ | วิธีแก้ |
|:--|:--|:--|
| `NameError` ในขั้นตอนหลัง ๆ | ข้ามขั้นตอนก่อนหน้า หรือเริ่มเซสชันใหม่ | Runtime ▸ Run all |
| ดาวน์โหลด Kaggle ล้มเหลว | เครือข่ายถูกจำกัด | เลือกแหล่งข้อมูล "ZIP จาก Google Drive" |
| ขั้นตอนที่ 4–5 ช้ามาก | ยังไม่ได้เปิด GPU | Runtime ▸ Change runtime type ▸ T4 GPU |
| ความแม่นยำต่ำ/ผันผวน | ภาพจริงน้อย ชุดทดสอบเล็ก | เพิ่มภาพจริง (≥ 50 ภาพ/คลาส), เพิ่ม Epochs, LR 0.0005 |
| Train สูงแต่ Val/Test ต่ำมาก | Overfitting หรือเพิ่มภาพมากเกินไป | ลด `target_per_class`, เปิด augment ระหว่างฝึก, เพิ่มภาพจริง |
| ขึ้นคำเตือน "โหลดน้ำหนัก ImageNet ไม่ได้" | อินเทอร์เน็ตมีปัญหา | ตรวจเครือข่าย แล้วรันขั้นตอนที่ 4 ใหม่ (ห้ามใช้ผลที่ได้จากน้ำหนักสุ่ม) |

### 👩‍🏫 หมายเหตุสำหรับวิทยากร
- **ภาพจริงสำคัญกว่าภาพที่สร้างเพิ่ม:** เป้าหมายขั้นต่ำ 50 ภาพจริงต่อคลาส; ถ่ายหลายแปลง หลายวัน หลายสภาพแสง และมีผู้เชี่ยวชาญกำกับป้ายชื่อ (ยืนยันด้วยค่าที่วัดจริง เช่น pH, N-P-K หรือการเพาะแยกเชื้อ)
- **ป้องกันข้อมูลรั่ว:** ถ้าภาพชุดเดียวกันมาจากแปลงเดียวกัน ให้จัดโฟลเดอร์ `train/` และ `val/` แยกตามแปลงไว้ใน ZIP แล้วเลือก "ใช้การแบ่ง train/val เดิม"
- ชื่อโฟลเดอร์คลาสควรเป็นอักษรอังกฤษ (กราฟใช้เลขคลาสเพื่อเลี่ยงปัญหาฟอนต์ไทย)
- ทดสอบรันครบทุกขั้นตอนบน Colab หนึ่งรอบก่อนวันงานจริง และสำรอง ZIP ไว้ในแฟลชไดรฟ์
- ผู้เรียนจำนวนมากใช้ Colab ฟรีพร้อมกันอาจติดโควตา GPU — ข้ามขั้นตอนที่ 5 และใช้ MobileNetV3 Small

*ชีวะ ทัศนา สาขาวิชาดิจิทัลการเกษตรและสิ่งแวดล้อม คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยราชภัฏรำไพพรรณี*